<a href="https://colab.research.google.com/github/eljaysmithdata/Data-Science-Bootcamp-2026/blob/main/Project-5/Project_5_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 5: Natural Language Processing

AI Disclosure: ChatGPT was used to support concept clarification, wording refinement, visualization formatting, and code execution.

In this project, I use Natural Language Processing to compare famous people based on written descriptions.

My guiding analogy is:

**NLP is like hearing meaning inside the noise.**

Instead of listening to audio, the model listens to language.  
It looks for patterns in words, filters out common background noise, and finds which descriptions carry a similar signal.

The goal is not to make the computer “understand” people like a human would.  
The goal is to turn text into a structure the computer can compare.

### Imports and Installs


In [ ]:
# ============================================================
# Import Libraries
# ============================================================

# Core data tools
import pandas as pd
import numpy as np

# Visualization tools
import matplotlib.pyplot as plt
import seaborn as sns

# Text processing and NLP tools
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Sentiment analysis
from textblob import TextBlob

# Display settings
pd.set_option("display.max_colwidth", 150)
pd.set_option('display.max_rows', None)

from IPython.display import display, clear_output

from google.colab import output
output.enable_custom_widget_manager()

!pip install textblob
!pip install wikipedia
!pip install ipywidgets -q

import wikipedia
import ipywidgets as widgets
from IPython.display import display

# Part 1: Overview-Based Similarity

For Part 1, I use the provided dataset of names and short descriptions.

Each description is like a short audio clip.

The process is:

1. Load the data  
2. Check the structure  
3. Choose one reference person  
4. Convert the descriptions into numerical text signals  
5. Find the 10 people with the most similar signal  
6. Analyze the sentiment of the reference person’s description  

In sound engineering terms, this is like choosing one vocal track and asking:

**Which other tracks have the most similar frequency pattern?**

## Load Data

First, I load the CSV file from the project URL.

This dataset includes:

- a URL
- a person's name
- a short written description of the person

This is the raw soundboard before any signal processing happens.

In [ ]:
# ============================================================
# Load Dataset
# ============================================================

# Project dataset URL
url = "https://ddc-datascience.s3.amazonaws.com/Projects/Project.5-NLP/Data/NLP.csv"

# Load CSV into dataframe
df = pd.read_csv(url)

# Preview the first few rows
df.head()

## Initial Data Check

Before modeling, I check the structure of the dataset.

This helps me understand:

- how many rows are available
- whether any values are missing
- which columns are included
- which column contains the text for NLP analysis

This is like checking the levels before recording.  
If the input is messy, the output will be messy too.

In [ ]:
# ============================================================
# Initial Data Check
# ============================================================

# Check the size of the dataset
print("Dataset shape:", df.shape)

# Check column names and data types
df.info()

In [ ]:
# ============================================================
# Check Missing Values
# ============================================================

# Count missing values in each column
df.isnull().sum()

In [ ]:
# ============================================================
# Preview Key Columns
# ============================================================

# Preview the name and text columns for the first 10 columns
df[["name", "text"]].head(11)

## Clean Working Dataset

Next, I create a clean working version of the dataset.

I keep the original dataframe untouched, then remove rows that are missing the name or description.

This is not dramatic cleaning.  
It is basic signal hygiene.

In [ ]:
# ============================================================
# Create Clean Working Dataset
# ============================================================



# Make a copy so the original dataframe stays unchanged
df_clean = df.copy()

# Remove rows missing the person name or description text
df_clean = df_clean.dropna(subset=["name", "text"]).reset_index(drop=True)

# Make sure the text column is stored as string data
df_clean["text"] = df_clean["text"].astype(str)

# Confirm clean dataset shape
print("Clean dataset shape:", df_clean.shape)

# Preview clean data
df_clean[["name", "text"]].head()

## Choose Reference Person

I chose **Dido** as my reference person.

Her description becomes the main signal.

The model will compare every other description against this one and find the closest matches based on word patterns.

Important note:  
This does not mean the people are the same.  
It means the language used to describe them has a similar pattern.

In [ ]:
# ============================================================
# Choose Reference Person
# ============================================================

# Set reference person
reference_person = "Dido"

# Find matching row
reference_match = df_clean[df_clean["name"].str.contains(reference_person, case=False, na=False)]

# Display match
reference_match[["name", "text"]]

In [ ]:
#Alternative solution is to create a function to search name:
#def search_person(name_part):
  #return df[df['name'].str.contains(name_part, case=False, na=False)][['name']]

In [ ]:
# ============================================================
# Confirm Reference Person
# ============================================================

# Store the exact dataset name and index
reference_name = reference_match["name"].iloc[0]
reference_index = reference_match.index[0]

print("Reference person:", reference_name)
print("Reference index:", reference_index)

In [ ]:
# ============================================================
# Select Reference Person Text
# ============================================================

reference_name = "Dido (singer)"  # update this if the dataset uses a longer name

dido_text = df.loc[df["name"] == reference_name, "text"].iloc[0]

print(dido_text)

In [ ]:
# ============================================================
# Count Word Frequencies
# ============================================================

vectorizer = CountVectorizer(
    stop_words="english",
    lowercase=True
)

word_count_matrix = vectorizer.fit_transform([dido_text])

word_counts = pd.DataFrame({
    "word": vectorizer.get_feature_names_out(),
    "count": word_count_matrix.toarray()[0]
})

word_counts = word_counts.sort_values("count", ascending=False)

word_counts.head(15)

In [ ]:
# ============================================================
# Plot Top Word Frequencies
# ============================================================

top_words = word_counts.head(10).sort_values("count")

plt.figure(figsize=(8, 5))
plt.barh(top_words["word"], top_words["count"])

plt.xlabel("Frequency")
plt.ylabel("Word")
plt.title("Most Frequent Words in Dido's DBpedia Overview")

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Display Dido's DBpedia Row
# ============================================================

dido_row = df[df["name"].str.contains("Dido", case=False, na=False)][["name", "text"]]

display(dido_row)

In [ ]:
# ============================================================
# Clean Display for Slide Screenshot
# ============================================================

pd.set_option("display.max_colwidth", 44)

dido_row = df[df["name"].str.contains("Dido", case=False, na=False)][["name", "text"]]

display(dido_row)

## Convert Text into TF-IDF Signals

Now I convert the written descriptions into numbers using TF-IDF.

TF-IDF stands for **Term Frequency-Inverse Document Frequency**.

Plain English:

TF-IDF gives more weight to words that are meaningful in one description but not overly common across the whole dataset.

Sound engineering analogy:

This lowers the background hum and brings the more distinctive frequencies forward.

In [ ]:
# ============================================================
# Convert Text to TF-IDF Features
# ============================================================

# Create TF-IDF vectorizer
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

# Transform description text into numerical signals
tfidf_matrix = tfidf.fit_transform(df_clean["text"])

# Check matrix shape
print("TF-IDF matrix shape:", tfidf_matrix.shape)

## Find Similar Descriptions

Next, I use cosine similarity to compare Dido's text signal to everyone else's.

Cosine similarity does not ask, "Who has the most words?"

It asks:

**Which text signal points in the most similar direction?**

That matters because longer descriptions should not automatically win just because they are louder.

In sound terms, this is not volume matching.  
This is signal-shape matching.

In [ ]:
# ============================================================
# Calculate Cosine Similarity
# ============================================================

# Compare reference person's TF-IDF vector to all other vectors
similarity_scores = cosine_similarity(
    tfidf_matrix[reference_index],
    tfidf_matrix
).flatten()

# Add similarity scores to dataframe
df_clean["similarity_to_reference"] = similarity_scores

# Remove the reference person from the results
similar_people = df_clean[df_clean.index != reference_index].copy()

# Sort by similarity score and select the top 10
top_10_similar = similar_people.sort_values(
    by="similarity_to_reference",
    ascending=False
).head(10)

# Display results
top_10_similar[["name", "similarity_to_reference", "text"]]

## Top 10 Closest People

Next, I describe the dataframe result more clearly.

The people listed below are the closest matches based on the word patterns in their descriptions.

This does not mean they are emotionally, artistically, or personally the same as Dido.

It means their descriptions share similar language signals.

Tiny distinction. Big deal.

In [ ]:
# ============================================================
# Display Top 10 Similar People
# ============================================================

# Store only the columns needed for interpretation
top_10_display = top_10_similar[
    ["name", "similarity_to_reference", "text"]
].reset_index(drop=True)

top_10_display

## Interpretation of Nearest Neighbors

The nearest neighbors include people whose descriptions share meaningful language patterns with Alanis Morissette's description.

The model is mostly responding to patterns in words related to music, performance, public identity, and career description.

This is useful, but it is also limited.

The model does not understand Alanis Morissette as a person.  
It understands the statistical shape of the words used to describe her.

That is the NLP lens here:

**The model is hearing repeated meaning patterns inside the text signal.**

## Sentiment Analysis of Reference Text

Next, I analyze the sentiment of the reference person's short dataset description.

I use TextBlob to calculate:

- **polarity**, which estimates whether the text sounds more negative, neutral, or positive
- **subjectivity**, which estimates whether the text sounds more factual or opinion-based

This is not a deep emotional reading.

It is more like a quick meter check on the tone of the text.

In [ ]:
# ============================================================
# Sentiment Analysis of Reference Description
# ============================================================

# Pull reference description
reference_text = df_clean.loc[reference_index, "text"]

# Analyze sentiment
sentiment = TextBlob(reference_text).sentiment

# Create simple sentiment label
if sentiment.polarity > 0:
    sentiment_label = "Positive"
elif sentiment.polarity < 0:
    sentiment_label = "Negative"
else:
    sentiment_label = "Neutral"

# Print results
print("Reference person:", reference_name)
print("Sentiment label:", sentiment_label)
print("Polarity:", sentiment.polarity)
print("Subjectivity:", sentiment.subjectivity)

## Sentiment Interpretation

The sentiment score gives a basic tone reading of the dataset description.

If the polarity is positive but close to zero, I would interpret that as slightly positive but still close to neutral.

If the subjectivity score is low to moderate, that suggests the description is written more like factual biography than personal opinion.

This matters because short biography descriptions are usually compressed and informational.

So the sentiment result is useful as a quick signal check, but I would not treat it as the whole emotional truth of the person.

A biography can be technically positive and still leave out the actual storm.

## Part 1 Summary

In Part 1, I used NLP to compare short biographical descriptions.

The process was:

1. Load the dataset  
2. Clean the name and text columns  
3. Choose Dido as the reference person  
4. Convert descriptions into TF-IDF vectors  
5. Use cosine similarity to find the 10 closest descriptions  
6. Use TextBlob to analyze sentiment  

The nearest neighbors show which descriptions have the most similar language patterns.

In terms of the sound engineering analogy:

**TF-IDF shaped the signal.**  
**Cosine similarity helped compare the shape of the signal.**  
**The nearest neighbors were the tracks with the most similar language mix.**

# Part 2: Wikipedia API and Full-Text Similarity

In Part 1, I compared people using short dataset descriptions.

In Part 2, I use the Wikipedia API to collect longer article text for the same reference person and their 10 nearest neighbors.

This lets me compare two kinds of text samples:

- short dataset descriptions
- longer Wikipedia articles

Sound engineering analogy:

Part 1 used short clips.  
Part 2 uses fuller recordings.

The question is whether the longer recordings still carry similar signals.

In [ ]:
# ============================================================
# Install and Import Wikipedia API
# ============================================================

!pip install wikipedia-api -q

import wikipediaapi

In [ ]:
# ============================================================
# Connect to Wikipedia
# ============================================================

# Create Wikipedia object with a user agent
wiki = wikipediaapi.Wikipedia(
    language="en",
    user_agent="NLP Project Student Notebook"
)

## Retrieve the Reference Person's Wikipedia Article

Next, I use the Wikipedia API to retrieve the full article for the reference person.

This expands the signal from a short description into a longer body of text.

In [ ]:
# ============================================================
# Retrieve Reference Wikipedia Article
# ============================================================

# Get Wikipedia page for reference person
reference_page = wiki.page(reference_name)

# Check whether page exists
print("Page exists:", reference_page.exists())
print("Page title:", reference_page.title)

# Store article text
reference_wiki_text = reference_page.text

# Preview first 1000 characters
print(reference_wiki_text[:1000])

## Sentiment of the Wikipedia Article

Now I calculate sentiment on the full Wikipedia article.

Because this article is much longer than the dataset description, the sentiment result may be different.

A longer article can include career details, awards, controversy, personal history, and context.

Sound engineering analogy:

The short description is a sample clip.  
The Wikipedia article is closer to the full track.

In [ ]:
# ============================================================
# Sentiment Analysis of Reference Wikipedia Article
# ============================================================

# Analyze Wikipedia article sentiment
wiki_sentiment = TextBlob(reference_wiki_text).sentiment

# Create simple sentiment label
if wiki_sentiment.polarity > 0:
    wiki_sentiment_label = "Positive"
elif wiki_sentiment.polarity < 0:
    wiki_sentiment_label = "Negative"
else:
    wiki_sentiment_label = "Neutral"

# Print results
print("Reference person:", reference_name)
print("Wikipedia sentiment label:", wiki_sentiment_label)
print("Wikipedia polarity:", wiki_sentiment.polarity)
print("Wikipedia subjectivity:", wiki_sentiment.subjectivity)

## Wikipedia Sentiment Interpretation

The Wikipedia sentiment score gives a broader tone reading of the reference person's full article.

Compared with the dataset description, the Wikipedia page contains more context and more variation in language.

This means the sentiment may be less simple than the short overview.

In sound engineering terms:

The short description is a clean sample.  
The full Wikipedia article is the whole mix.

## Collect Wikipedia Articles for the 10 Nearest Neighbors

Next, I collect Wikipedia article text for the 10 closest people from Part 1.

This lets me compare the reference person's full Wikipedia article to the full Wikipedia articles for the nearest neighbors.

Now the comparison moves from short description clips to longer article tracks.

In [ ]:
# ============================================================
# Collect Wikipedia Articles for Nearest Neighbors
# ============================================================

# Store nearest neighbor names
neighbor_names = top_10_similar["name"].tolist()

# Create list to hold Wikipedia article data
wiki_articles = []

# Loop through each nearest neighbor
for person in neighbor_names:

    # Get Wikipedia page
    page = wiki.page(person)

    # Only keep pages that exist and have text
    if page.exists() and len(page.text) > 0:
        wiki_articles.append({
            "name": person,
            "wiki_title": page.title,
            "wiki_text": page.text
        })

# Convert results into dataframe
wiki_df = pd.DataFrame(wiki_articles)

# Preview collected Wikipedia articles
wiki_df[["name", "wiki_title"]]

## Convert Wikipedia Articles into Numerical Features

Now I convert the Wikipedia article text into numerical features using TF-IDF.

This is the same basic method from Part 1, but the input text is longer.

That matters because longer articles can carry more context, but they can also introduce more noise.

Sound engineering analogy:

The dataset descriptions were short clips.  
The Wikipedia articles are longer tracks with more instruments in the mix.

In [ ]:
# ============================================================
# Convert Wikipedia Articles to TF-IDF Features
# ============================================================

# Create a row for the reference person's Wikipedia article
reference_wiki_row = pd.DataFrame([{
    "name": reference_name,
    "wiki_title": reference_page.title,
    "wiki_text": reference_wiki_text
}])

# Combine the reference article with the nearest-neighbor articles
wiki_comparison_df = pd.concat(
    [reference_wiki_row, wiki_df],
    ignore_index=True
)

# Remove rows with missing or empty Wikipedia text
wiki_comparison_df = wiki_comparison_df.dropna(subset=["wiki_text"]).reset_index(drop=True)
wiki_comparison_df = wiki_comparison_df[wiki_comparison_df["wiki_text"].str.len() > 0].reset_index(drop=True)

# Create TF-IDF vectorizer for Wikipedia article text
wiki_tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=5000
)

# Transform Wikipedia text into numerical features
wiki_tfidf_matrix = wiki_tfidf.fit_transform(wiki_comparison_df["wiki_text"])

# Check matrix shape
print("Wikipedia TF-IDF matrix shape:", wiki_tfidf_matrix.shape)

## Measure Wikipedia Article Similarity

Next, I use cosine similarity again.

This time, I compare the reference person's full Wikipedia article to the Wikipedia articles for the 10 nearest neighbors from Part 1.

This asks a slightly different question:

Part 1 asked:

**Who has the most similar short description?**

Part 2 asks:

**Do the longer Wikipedia articles still carry a similar signal?**

In [ ]:
# ============================================================
# Measure Wikipedia Article Similarity
# ============================================================

# The reference person is the first row in wiki_comparison_df
wiki_reference_index = 0

# Compare the reference Wikipedia article to all other Wikipedia articles
wiki_similarity_scores = cosine_similarity(
    wiki_tfidf_matrix[wiki_reference_index],
    wiki_tfidf_matrix
).flatten()

# Add Wikipedia similarity scores to the dataframe
wiki_comparison_df["wiki_similarity_to_reference"] = wiki_similarity_scores

# Remove the reference person from the displayed results
wiki_similarity_results = wiki_comparison_df[
    wiki_comparison_df.index != wiki_reference_index
].copy()

# Sort by Wikipedia similarity score
wiki_similarity_results = wiki_similarity_results.sort_values(
    by="wiki_similarity_to_reference",
    ascending=False
).reset_index(drop=True)

# Display results
wiki_similarity_results[[
    "name",
    "wiki_title",
    "wiki_similarity_to_reference"
]]

## Rank Neighbors by Wikipedia Similarity

Now I rank the nearest neighbors based on Wikipedia article similarity.

This ranking may be different from Part 1 because the model is now comparing fuller text signals.

That is not a problem.

It just means the input changed, so the signal changed.

In [ ]:
# ============================================================
# Rank Neighbors by Wikipedia Similarity
# ============================================================

# Add Wikipedia rank
wiki_similarity_results["wiki_rank"] = range(1, len(wiki_similarity_results) + 1)

# Create cleaner ranking table
wiki_ranking = wiki_similarity_results[[
    "wiki_rank",
    "name",
    "wiki_title",
    "wiki_similarity_to_reference"
]]

wiki_ranking

## Compare Part 1 Ranking with Wikipedia Ranking

Now I compare the original dataset ranking with the Wikipedia article ranking.

A small rank difference means the person stayed similarly close across both text sources.

A larger rank difference means the similarity changed when more context was added.

Sound engineering analogy:

A short clip might sound similar, but the full track can reveal a different mix.

In [ ]:
# ============================================================
# Compare Dataset Ranking and Wikipedia Ranking
# ============================================================

# Create Part 1 ranking table
dataset_ranking = top_10_similar[[
    "name",
    "similarity_to_reference"
]].copy()

dataset_ranking["dataset_rank"] = range(1, len(dataset_ranking) + 1)

# Create Wikipedia ranking table
wiki_ranking_for_merge = wiki_similarity_results[[
    "name",
    "wiki_similarity_to_reference",
    "wiki_rank"
]].copy()

# Merge rankings
ranking_comparison = dataset_ranking.merge(
    wiki_ranking_for_merge,
    on="name",
    how="left"
)

# Calculate rank difference
ranking_comparison["rank_difference"] = (
    ranking_comparison["dataset_rank"] - ranking_comparison["wiki_rank"]
).abs()

# Sort by original dataset rank
ranking_comparison = ranking_comparison.sort_values(
    by="dataset_rank"
).reset_index(drop=True)

ranking_comparison

## Visualize Rank Differences

This chart shows how much each person's rank changed between the short dataset description and the longer Wikipedia article.

A lower bar means the ranking stayed more stable.

A higher bar means the longer article changed the similarity relationship more strongly.

This is where the project gets interesting:

The model is not giving one eternal answer.  
It is responding to the text sample we give it.

In [ ]:
# ============================================================
# Visualize Rank Differences
# ============================================================

plt.figure(figsize=(10, 6))

plt.barh(
    ranking_comparison["name"],
    ranking_comparison["rank_difference"]
)

plt.xlabel("Absolute Rank Difference")
plt.ylabel("Person")
plt.title("Difference Between Dataset Ranking and Wikipedia Ranking")

plt.gca().invert_yaxis()

plt.show()

## Part 2 Summary

In Part 2, I expanded the analysis from short descriptions to full Wikipedia articles.

The process was:

1. Retrieve Wikipedia article text for the reference person  
2. Retrieve Wikipedia article text for the 10 nearest neighbors  
3. Convert Wikipedia articles into TF-IDF vectors  
4. Measure article similarity using cosine similarity  
5. Compare the original dataset ranking with the Wikipedia ranking  

The comparison shows how sensitive NLP results are to the text source.

In sound engineering terms:

Part 1 compared short clips.  
Part 2 compared fuller tracks.

The signal became richer, but also messier.

# Part 3: Interactive NLP Notebook

In Part 3, I turn the notebook into an interactive NLP tool.

Instead of only using one fixed reference person, the user can choose a person from the dataset.

The notebook then returns:

- the selected person's sentiment score
- the closest text matches
- the similarity scores for those matches

Sound engineering analogy:

Part 1 chose one reference track.  
Part 3 lets the user choose the reference track and immediately hear how the mix changes.

In [ ]:
# ============================================================
# Install and Import Interactive Tools
# ============================================================

!pip install ipywidgets -q

import ipywidgets as widgets
from IPython.display import display, clear_output

## Create Similarity Function

Rather than repeating the same code, I create a reusable function.

The function:

1. takes a person's name  
2. finds that person in the dataset  
3. calculates similarity scores  
4. returns the closest matches  

This keeps the notebook cleaner and makes the interactive part easier to run.

In [ ]:
# ============================================================
# Create Similarity Function
# ============================================================

def find_similar_people(person_name, top_n=10):
    """
    Finds the most similar people based on TF-IDF and cosine similarity.
    """

    # Find selected person's index
    selected_index = df_clean[df_clean["name"] == person_name].index[0]

    # Calculate similarity between selected person and everyone else
    selected_similarity_scores = cosine_similarity(
        tfidf_matrix[selected_index],
        tfidf_matrix
    ).flatten()

    # Create results dataframe
    results = df_clean.copy()
    results["similarity_score"] = selected_similarity_scores

    # Remove selected person from their own results
    results = results[results.index != selected_index]

    # Sort and return top matches
    results = results.sort_values(
        by="similarity_score",
        ascending=False
    ).head(top_n)

    return results[["name", "similarity_score", "text"]]

## Create Sentiment Function

Next, I create a reusable sentiment function.

This function takes a person's name and returns:

- the person's description
- polarity
- subjectivity
- a simple sentiment label

This is the quick tone meter for the selected text.

In [ ]:
# ============================================================
# Create Sentiment Function
# ============================================================

def analyze_person_sentiment(person_name):
    """
    Analyzes sentiment for a selected person's dataset description.
    """

    # Find selected person's index
    selected_index = df_clean[df_clean["name"] == person_name].index[0]

    # Pull selected text
    selected_text = df_clean.loc[selected_index, "text"]

    # Analyze sentiment
    selected_sentiment = TextBlob(selected_text).sentiment

    # Create sentiment label
    if selected_sentiment.polarity > 0:
        sentiment_label = "Positive"
    elif selected_sentiment.polarity < 0:
        sentiment_label = "Negative"
    else:
        sentiment_label = "Neutral"

    return selected_text, sentiment_label, selected_sentiment.polarity, selected_sentiment.subjectivity

## Create User Selection Tool

Now I create a dropdown menu so the user can choose a person from the dataset.

This makes the notebook interactive.

Instead of rewriting code, the user changes the input and the notebook updates the result.

In [ ]:
df
tfidf_matrix

In [ ]:
df
tfidf_matrix
vectorizer

In [ ]:
# ============================================================
# Check Required Objects
# ============================================================

print("df shape:", df.shape)
print("tfidf_matrix shape:", tfidf_matrix.shape)
print("Number of names:", df["name"].nunique())

In [ ]:
# ============================================================
# Function: Find Closest People
# ============================================================

from sklearn.metrics.pairwise import cosine_similarity
from textblob import TextBlob

def find_closest_people(reference_name, top_n=10):
    """
    Finds the closest people to the selected reference person
    using TF-IDF and cosine similarity.
    """

    # Search flexibly instead of requiring exact match
    matches = df[df["name"].str.contains(reference_name, case=False, na=False)]

    if matches.empty:
        print(f"No match found for: {reference_name}")
        print("Try a shorter search term or check spelling.")
        return

    # Use the first matching person
    reference_index = matches.index[0]
    exact_name = df.loc[reference_index, "name"]
    reference_text = df.loc[reference_index, "text"]

    print(f"Reference person used: {exact_name}")
    print()

    # Compare selected person to everyone else
    similarities = cosine_similarity(
        tfidf_matrix[reference_index],
        tfidf_matrix
    ).flatten()

    results = df.copy()
    results["similarity_score"] = similarities

    # Remove the reference person from their own results
    results = results[results.index != reference_index]

    # Return top matches
    top_matches = results.sort_values(
        by="similarity_score",
        ascending=False
    ).head(top_n)

    display(top_matches[["name", "similarity_score", "text"]])

    # Sentiment for reference person's overview
    sentiment = TextBlob(reference_text).sentiment

    print()
    print("Reference Overview Sentiment")
    print("Polarity:", sentiment.polarity)
    print("Subjectivity:", sentiment.subjectivity)

In [ ]:
# ============================================================
# Interactive Search Widget
# ============================================================

name_input = widgets.Text(
    value="Dido",
    placeholder="Type a famous person",
    description="Name:",
    layout=widgets.Layout(width="25%")
)

top_n_slider = widgets.IntSlider(
    value=10,
    min=1,
    max=20,
    step=1,
    description="Top N:"
)

search_button = widgets.Button(
    description="Find Similar People",
    button_style="primary"
)

output_area = widgets.Output()

def run_search(button):
    with output_area:
        clear_output()
        find_closest_people(name_input.value, top_n_slider.value)

search_button.on_click(run_search)

display(name_input, top_n_slider, search_button, output_area)

In [ ]:
# ============================================================
# Create User Selection Tool
# ============================================================

# Create dropdown menu
person_dropdown = widgets.Dropdown(
    options=sorted(df_clean["name"].unique()),
    value=reference_name,
    description="Person:"
)

# Create slider for number of results
top_n_slider = widgets.IntSlider(
    value=10,
    min=3,
    max=20,
    step=1,
    description="Top N:"
)

# Create output area
output = widgets.Output()

## Connect User Input to Similarity Function

Now I connect the dropdown menu and slider to the functions.

When the user changes the selected person, the notebook automatically updates:

- the overview text
- the sentiment result
- the nearest neighbors

This turns the notebook into a small NLP signal dashboard.

In [ ]:
# ============================================================
# Connect User Input to Similarity Function
# ============================================================

def update_dashboard(change=None):
    """
    Updates the interactive NLP dashboard.
    """

    with output:
        clear_output()

        # Get selected user inputs
        selected_person = person_dropdown.value
        top_n = top_n_slider.value

        # Run sentiment analysis
        selected_text, sentiment_label, polarity, subjectivity = analyze_person_sentiment(selected_person)

        # Find similar people
        similar_results = find_similar_people(selected_person, top_n)

        # Display selected person
        print("Reference signal:", selected_person)
        print("=" * 70)

        # Display overview
        print("\nOverview:")
        print(selected_text)

        # Display sentiment
        print("\nSentiment:")
        print("Label:", sentiment_label)
        print("Polarity:", polarity)
        print("Subjectivity:", subjectivity)

        # Display similar people
        print("\nClosest text signals:")
        display(similar_results)

# Update dashboard when dropdown or slider changes
person_dropdown.observe(update_dashboard, names="value")
top_n_slider.observe(update_dashboard, names="value")

# Display interactive tools
display(person_dropdown, top_n_slider, output)

# Run once when cell loads
update_dashboard()

## Part 3 Summary

In Part 3, I transformed the notebook into an interactive NLP tool.

The process was:

1. Create a reusable similarity function  
2. Create a reusable sentiment function  
3. Add a dropdown menu for selecting a person  
4. Add a slider for choosing the number of results  
5. Connect the user input to the NLP functions  

This allows users to explore the dataset without changing the code.

In sound engineering terms:

Part 1 cleaned and isolated the signal.  
Part 2 expanded the signal into fuller tracks.  
Part 3 lets the user move the dial.

# Final Reflection

This project helped me understand NLP as a form of signal processing for language.

The computer is not understanding biography like a human does.

It is turning text into patterns, comparing those patterns, and giving us a structured way to ask:

- Which descriptions use similar language?
- What tone does this text carry?
- How does the answer change when the text sample gets longer?

The most important lesson is that NLP results depend on the text sample.

A short overview and a full Wikipedia article can describe the same person but produce different results because they contain different levels of detail.

In sound engineering terms:

**The input changes the mix.**  
**The cleaning shapes the signal.**  
**The model finds patterns, not truth.**